# Logged training metrics

Show the classification metrics written by a training run to Lightning's CSV log.

## Question and success signal

Which train, validation, and test values are in the latest Lightning CSV log, and how did they change by epoch?

## Setup and configuration

Kernel: the repository environment (`uv run`, Python 3.14). Working directory: repository root or `notebooks/`. Input: `lightning_logs/metrics.csv` from `CSVLogger(save_dir=".", name="lightning_logs", version="")`.

In [4]:
from __future__ import annotations

import csv
from pathlib import Path

from IPython.display import Markdown, display

SPLITS: tuple[str, ...] = ("train", "val", "test")
METRIC_COLUMNS: tuple[tuple[str, str], ...] = (
    ("loss", "loss"),
    ("acc_top1", "top-1 accuracy"),
    ("acc_top5", "top-5 accuracy"),
    ("precision", "precision"),
    ("recall", "recall"),
    ("f1", "F1"),
)


def find_repo_root(start: Path) -> Path:
    """Return the repository root that contains train.py and utils/metrics.py.

    Args:
        start: Directory to walk upward from.

    Returns:
        The repository root path.

    Raises:
        FileNotFoundError: If no ancestor contains both marker files.
    """
    for candidate in (start, *start.parents):
        if (candidate / "train.py").is_file() and (
            candidate / "utils" / "metrics.py"
        ).is_file():
            return candidate
    message = (
        "Could not locate the repository root from the notebook working directory."
    )
    raise FileNotFoundError(message)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
LOG_DIR = REPO_ROOT / "lightning_logs"
display(Markdown(f"Repository root: `{REPO_ROOT}`"))

Repository root: `/home/darke/code/ResNet_Lightning`

## Inputs and assumptions

`train.py` logs epoch scalars named `{split}_{metric}`. Train and validation rows for the same epoch can appear separately; empty CSV cells are missing values, not zeros. `lightning_logs/` is gitignored, so this notebook needs a local training run.

In [5]:
def find_metrics_csv(log_dir: Path) -> Path | None:
    """Return the preferred metrics.csv, or the last nested match.

    Args:
        log_dir: Lightning CSV logger directory.

    Returns:
        Path to metrics.csv, or None if no log file exists.
    """
    preferred = log_dir / "metrics.csv"
    if preferred.is_file():
        return preferred
    if not log_dir.is_dir():
        return None
    matches = sorted(path for path in log_dir.rglob("metrics.csv") if path.is_file())
    return matches[-1] if matches else None


def merge_rows_by_epoch(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    """Fold CSV rows that share an epoch into one display row.

    Args:
        rows: Raw DictReader rows from Lightning metrics.csv.

    Returns:
        One mapping per epoch, last non-empty value wins per column.
    """
    merged: dict[str, dict[str, str]] = {}
    order: list[str] = []
    for row in rows:
        epoch = row.get("epoch", "").strip()
        key = epoch if epoch else f"step:{row.get('step', '')}"
        if key not in merged:
            merged[key] = {}
            order.append(key)
        bucket = merged[key]
        for name, value in row.items():
            text = value.strip()
            if text:
                bucket[name] = text
        if epoch:
            bucket["epoch"] = epoch
    return [merged[key] for key in order]


def markdown_table(headers: list[str], body: list[list[str]]) -> str:
    """Render a GitHub-style Markdown table.

    Args:
        headers: Column names.
        body: Row values aligned with ``headers``.

    Returns:
        Markdown table text.
    """
    header_line = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join("---" for _ in headers) + " |"
    lines = [header_line, separator]
    for row in body:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)


METRICS_CSV = find_metrics_csv(LOG_DIR)
if METRICS_CSV is None:
    ROWS: list[dict[str, str]] = []
    display(
        Markdown(
            f"No `metrics.csv` found under `{LOG_DIR}`. "
            "Run `uv run python train.py` to write Lightning CSV logs, then rerun this notebook."
        )
    )
else:
    with METRICS_CSV.open(newline="", encoding="utf-8") as handle:
        ROWS = merge_rows_by_epoch(list(csv.DictReader(handle)))
    display(
        Markdown(
            f"Loaded `{METRICS_CSV.relative_to(REPO_ROOT)}` ({len(ROWS)} epoch rows)."
        )
    )

Loaded `lightning_logs/metrics.csv` (10 epoch rows).

## Exploration

Display the latest epoch as a split-by-metric table, then the per-epoch history for each split that has values.

In [6]:
def format_metric(row: dict[str, str], split: str, suffix: str) -> str:
    """Format one logged scalar for display.

    Args:
        row: Merged epoch mapping.
        split: Log-name prefix such as ``train``.
        suffix: Metric suffix such as ``acc_top1``.

    Returns:
        Six-decimal float text, the raw token, or an em dash when missing.
    """
    value = row.get(f"{split}_{suffix}", "").strip()
    if not value:
        return "—"
    try:
        return f"{float(value):.6f}"
    except ValueError:
        return value


def latest_table(rows: list[dict[str, str]]) -> str:
    """Build the latest-epoch split comparison table.

    Args:
        rows: Merged epoch mappings in log order.

    Returns:
        Markdown for the last epoch, or a missing-data message.
    """
    if not rows:
        return "No metric rows to display."
    row = rows[-1]
    epoch = row.get("epoch", "").strip() or "unknown"
    headers = ["Metric", *[split.capitalize() for split in SPLITS]]
    body = [
        [label, *[format_metric(row, split, suffix) for split in SPLITS]]
        for suffix, label in METRIC_COLUMNS
    ]
    return f"Latest epoch: **{epoch}**\n\n" + markdown_table(headers, body)


def history_table(rows: list[dict[str, str]], split: str) -> str:
    """Build one split's per-epoch metric history.

    Args:
        rows: Merged epoch mappings in log order.
        split: Log-name prefix such as ``train``.

    Returns:
        Markdown table of epoch versus metrics.
    """
    headers = ["epoch", *[label for _, label in METRIC_COLUMNS]]
    body = [
        [
            row.get("epoch", "").strip() or "—",
            *[format_metric(row, split, suffix) for suffix, _ in METRIC_COLUMNS],
        ]
        for row in rows
    ]
    return markdown_table(headers, body)


def split_has_values(rows: list[dict[str, str]], split: str) -> bool:
    """Return whether any row has a value for the split.

    Args:
        rows: Merged epoch mappings.
        split: Log-name prefix such as ``train``.

    Returns:
        True when at least one metric cell is non-empty.
    """
    return any(
        row.get(f"{split}_{suffix}", "").strip()
        for row in rows
        for suffix, _ in METRIC_COLUMNS
    )


if not ROWS:
    display(Markdown("No metric values to show."))
else:
    display(Markdown("### Latest epoch"))
    display(Markdown(latest_table(ROWS)))
    for split in SPLITS:
        if split_has_values(ROWS, split):
            display(Markdown(f"### {split.capitalize()} history"))
            display(Markdown(history_table(ROWS, split)))

### Latest epoch

Latest epoch: **9**

| Metric | Train | Val | Test |
| --- | --- | --- | --- |
| loss | 0.078101 | 1.585772 | — |
| top-1 accuracy | 0.989132 | 0.654250 | — |
| top-5 accuracy | 0.998868 | 0.857583 | — |
| precision | 0.989358 | 0.665561 | — |
| recall | 0.989132 | 0.654250 | — |
| F1 | 0.989130 | 0.651316 | — |

### Train history

| epoch | loss | top-1 accuracy | top-5 accuracy | precision | recall | F1 |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | 5.837497 | 0.034485 | 0.111779 | 0.033035 | 0.034485 | 0.029911 |
| 1 | 4.006800 | 0.180279 | 0.406353 | 0.158276 | 0.180279 | 0.162710 |
| 2 | 2.933760 | 0.337853 | 0.618706 | 0.315229 | 0.337853 | 0.321684 |
| 3 | 2.292219 | 0.450412 | 0.733676 | 0.433828 | 0.450412 | 0.438616 |
| 4 | 1.850384 | 0.536162 | 0.809250 | 0.523527 | 0.536162 | 0.527318 |
| 5 | 1.499898 | 0.609456 | 0.860809 | 0.600285 | 0.609456 | 0.602968 |
| 6 | 1.052058 | 0.713912 | 0.919059 | 0.708827 | 0.713912 | 0.709931 |
| 7 | 0.574731 | 0.838574 | 0.968897 | 0.836878 | 0.838574 | 0.836878 |
| 8 | 0.221821 | 0.945368 | 0.993456 | 0.945509 | 0.945368 | 0.945079 |
| 9 | 0.078101 | 0.989132 | 0.998868 | 0.989358 | 0.989132 | 0.989130 |

### Val history

| epoch | loss | top-1 accuracy | top-5 accuracy | precision | recall | F1 |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | 4.803255 | 0.081833 | 0.243917 | 0.071107 | 0.081833 | 0.052021 |
| 1 | 3.423456 | 0.267250 | 0.531000 | 0.297582 | 0.267250 | 0.235242 |
| 2 | 2.914272 | 0.349750 | 0.626167 | 0.412913 | 0.349750 | 0.325795 |
| 3 | 2.451644 | 0.434333 | 0.708167 | 0.500013 | 0.434333 | 0.414832 |
| 4 | 2.441964 | 0.444833 | 0.715000 | 0.521122 | 0.444833 | 0.434379 |
| 5 | 2.230206 | 0.487083 | 0.754750 | 0.568708 | 0.487083 | 0.476596 |
| 6 | 2.052347 | 0.531583 | 0.785917 | 0.607804 | 0.531583 | 0.526075 |
| 7 | 1.919949 | 0.572417 | 0.813333 | 0.614642 | 0.572417 | 0.565649 |
| 8 | 1.671874 | 0.631583 | 0.847250 | 0.652728 | 0.631583 | 0.627313 |
| 9 | 1.585772 | 0.654250 | 0.857583 | 0.665561 | 0.654250 | 0.651316 |

## Observations and next decision

If the CSV is missing, train with `uv run python train.py` and rerun this notebook. If the CSV is present, use the latest-epoch table as the run summary.